# Course 5: Building a Small Bangla Language Model
## Session 4: Tokenizers for Bangla

**Goal:** train BPE tokenizers at several sizes and choose one with measurements.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every code cell here was run once on a CPU (November 2026) using the small built-in sample. Cells marked TODO or NOT RUN need your data, a GPU or the network.

In [ ]:
# On Colab or Kaggle, run once:  !pip install tokenizers

In [ ]:
SAMPLE_LINES = [
    "আমি বাংলায় গান গাই।",
    "আমার সোনার বাংলা, আমি তোমায় ভালোবাসি।",
    "বাংলা আমার মাতৃভাষা।",
    "আজ আকাশ খুব পরিষ্কার।",
    "ছোট মেয়েটি নদীর ধারে খেলছিল।",
    "বাবা বাজার থেকে মাছ কিনে আনলেন।",
    "মা রান্নাঘরে ভাত রান্না করছেন।",
    "আমরা সবাই মিলে স্কুলে যাই।",
    "বৃষ্টি হলে মাঠে পানি জমে।",
    "পাখিরা সকালে গান গায়।",
    "দাদু প্রতিদিন সন্ধ্যায় গল্প বলেন।",
    "ছেলেটি বই পড়তে খুব ভালোবাসে।",
    "নদীতে অনেক নৌকা চলছে।",
    "শীতকালে আমরা পিঠা খাই।",
    "ঢাকা বাংলাদেশের রাজধানী।",
    "আমার বন্ধু ভালো ফুটবল খেলে।",
]

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders, normalizers

def train_bpe(lines, vocab_size, specials=("[UNK]", "<|endoftext|>")):
    tok = Tokenizer(models.BPE(unk_token="[UNK]"))
    tok.normalizer = normalizers.NFC()
    tok.pre_tokenizer = pre_tokenizers.Metaspace()
    tok.decoder = decoders.Metaspace()
    trainer = trainers.BpeTrainer(vocab_size=vocab_size, special_tokens=list(specials))
    tok.train_from_iterator(lines, trainer)
    return tok

### 1. Three ways to cut one sentence

In [ ]:
s = "আমি বাংলায় গান গাই"
print("bytes:", len(s.encode("utf-8")), "| code points:", len(s), "| words:", len(s.split()))

### 2. Train tokenizers
Replace `train_lines` and `val_lines` with lines from your `train.txt` and `val.txt`, and use sizes 2000, 8000 and 16000.

In [ ]:
train_lines = SAMPLE_LINES[:12] * 20      # TODO: open("train.txt", encoding="utf-8").read().splitlines()
val_lines = SAMPLE_LINES[12:]             # TODO: lines of val.txt
SIZES = [100, 200, 400]                   # TODO: [2000, 8000, 16000]

def tokens_per_word(tok, texts):
    return sum(len(tok.encode(t).ids) for t in texts) / sum(len(t.split()) for t in texts)

def tokens_per_char(tok, texts):
    return sum(len(tok.encode(t).ids) for t in texts) / sum(len(t) for t in texts)

toks = {}
for size in SIZES:
    toks[size] = train_bpe(train_lines, size)
    t = toks[size]
    mismatches = sum(t.decode(t.encode(x).ids) != x for x in val_lines)
    print(f"vocab {t.get_vocab_size():>6} | tokens/word {tokens_per_word(t, val_lines):.2f} | "
          f"tokens/char {tokens_per_char(t, val_lines):.2f} | round-trip mismatches {mismatches} | "
          f"embedding params at d=256: {t.get_vocab_size() * 256:,}")

On the tiny sample, the round-trip mismatches come from characters in the validation lines that never appear in training, so they become `[UNK]`. The vocabulary also stops growing once every merge is used up. Neither happens with a real corpus.

### 3. Look at the vocabulary

In [ ]:
t = toks[SIZES[-1]]
vocab = sorted(t.get_vocab(), key=len, reverse=True)
print("longest tokens:", vocab[:15])
print(t.encode("আমি বাংলায় গান গাই").tokens)

### 4. Compare with a borrowed tokenizer (TODO, NOT RUN)
Needs the network.

In [ ]:
# from transformers import AutoTokenizer
# other = AutoTokenizer.from_pretrained("PUT-A-MULTILINGUAL-MODEL-NAME-HERE")
# n_tok = sum(len(other.encode(x, add_special_tokens=False)) for x in val_lines)
# print("tokens/word:", n_tok / sum(len(x.split()) for x in val_lines))

### 5. Save your choice

In [ ]:
toks[SIZES[-1]].save("tokenizer.json")
print("saved tokenizer.json")